In [5]:
import os

os.environ["HF_HOME"] = r"D:\huggingface-cache"

print(os.environ["HF_HOME"])

D:\huggingface-cache


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [7]:
model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(model_name)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

summarizer_model.to(device)

print("Device:", device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

d:\Anaconda\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aniket\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Device: cuda


In [11]:
print("Tokenizer max length:", tokenizer.model_max_length)

Tokenizer max length: 1024


In [12]:
print("Model max position embeddings:",
      summarizer_model.config.max_position_embeddings)

Model max position embeddings: 1024


In [8]:
def summarize_text(text, max_length=100, min_length=30):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    summary_ids = summarizer_model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [23]:
sample_text = """
The European Union has announced a major investment programme aimed at
accelerating the development of artificial intelligence across the region.
The initiative will provide funding for AI startups, universities and
research institutions, with the goal of strengthening Europe's position
in the global technology industry.

Officials said the programme will focus on developing reliable and
energy-efficient AI systems. Funding will also support the creation of
new computing infrastructure that researchers and companies can use to
train large artificial intelligence models.

The European Commission said that AI could significantly improve
healthcare, transportation, manufacturing and public services. However,
officials also stressed the importance of ensuring that AI systems are
developed responsibly and that citizens' privacy and safety are protected.

Technology companies welcomed the additional investment, saying that
access to computing resources and research funding could help European
startups compete with larger companies in the United States and Asia.
"""

summary = summarize_text(
    sample_text,
    max_length=60,
    min_length=20
)

print(summary)

 Initiative will provide funding for AI startups, universities and research institutions . Aim is to strengthen Europe's position in the global technology industry . Funding will also support the creation of new computing infrastructure for large AI models .


## Steps to allow model to process long inputs

In [24]:
def split_text_into_chunks(text, max_words=400):
    """
    Split long text into smaller word-based chunks.
    """
    words = text.split()

    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)

    return chunks

In [26]:
sample_long_text = sample_text * 5

chunks = split_text_into_chunks(
    sample_long_text,
    max_words=400
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk.split())} words")

Number of chunks: 2
Chunk 1: 400 words
Chunk 2: 325 words


In [30]:
def summarize_long_text(
    text,
    chunk_size=400,
    chunk_max_length=80,
    chunk_min_length=30,
    final_max_length=120,
    final_min_length=50
):
    """
    Summarize long text using a two-stage approach:
    1. Summarize individual chunks.
    2. Summarize the combined chunk summaries.
    """

    chunks = split_text_into_chunks(
        text,
        max_words=chunk_size
    )

    chunk_summaries = []

    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i + 1} of {len(chunks)}...")

        summary = summarize_text(
            chunk,
            max_length=chunk_max_length,
            min_length=chunk_min_length
        )

        chunk_summaries.append(summary)

    combined_summary = " ".join(chunk_summaries)

    # If there is only one chunk, return its summary
    if len(chunks) == 1:
        return combined_summary

    print("Creating final summary...")

    final_summary = summarize_text(
        combined_summary,
        max_length=final_max_length,
        min_length=final_min_length
    )

    return final_summary

In [31]:
long_summary = summarize_long_text(
    sample_long_text,
    chunk_size=400,
    chunk_max_length=80,
    chunk_min_length=30,
    final_max_length=100,
    final_min_length=40
)

print("\nFINAL SUMMARY:")
print(long_summary)

Summarizing chunk 1 of 2...
Summarizing chunk 2 of 2...
Creating final summary...

FINAL SUMMARY:
 The European Union has announced a major investment programme aimed at accelerating the development of artificial intelligence across the region . The initiative will provide funding for AI startups, universities and research institutions . Officials said the programme will focus on developing reliable and energy-efficient AI systems .


In [32]:
print("Original words:", len(sample_long_text.split()))
print("Summary words:", len(long_summary.split()))

Original words: 725
Summary words: 48


From the above output, it can be seen that the summarization is working pretty well.

In [36]:
from src.summarizer import summarize_long_text

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [37]:
test_article = """
Artificial intelligence is transforming the way people consume news.
News organizations are increasingly using machine learning to categorize
large volumes of articles and identify important topics. Natural language
processing can also help summarize lengthy articles, allowing readers to
quickly understand the most important information without reading the
entire article. These technologies are becoming increasingly important
as the amount of digital information continues to grow.
"""

In [38]:
summary = summarize_long_text(
    test_article,
    chunk_size=400,
    final_max_length=80,
    final_min_length=30
)

print(summary)

 News organizations are increasingly using machine learning to categorize large volumes of articles and identify important topics . Natural language processing can also help summarize lengthy articles, allowing readers to quickly understand the most important information without reading the entire article .
